In [ ]:
import os
import json
import glob
import argparse
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configure plot style for publication-quality charts
plt.style.use('seaborn-v0_8-whitegrid')

class OSWorldAnalyzer:
    def __init__(self, run_dir):
        self.run_dir = run_dir
        self.data = []
        self.report_path = os.path.join(run_dir, "analysis_report.md")

    def load_data(self):
        """
        Crawls the run directory looking for trajectory.json files.
        Aggregates them into a Pandas DataFrame.
        """
        print(f"🔍 Scanning {self.run_dir} for trajectories...")
        # Recursive search for trajectory.json files
        files = glob.glob(os.path.join(self.run_dir, "**", "trajectory.json"), recursive=True)
        
        if not files:
            print("❌ No trajectory files found.")
            return

        for file_path in files:
            with open(file_path, 'r') as f:
                try:
                    entry = json.load(f)
                    # Heuristic Failure Analysis
                    failure_reason = self._diagnose_failure(entry)
                    
                    row = {
                        "task_id": entry.get("task_id"),
                        "domain": entry.get("domain"),
                        "success": float(entry.get("success_score", 0)) > 0,
                        "steps": entry.get("steps_taken"),
                        "instruction": entry.get("instruction"),
                        "failure_reason": failure_reason
                    }
                    self.data.append(row)
                except json.JSONDecodeError:
                    print(f"⚠️ Corrupt JSON: {file_path}")

        self.df = pd.DataFrame(self.data)
        print(f"✅ Loaded {len(self.df)} tasks.")

    def _diagnose_failure(self, entry):
        """
        Analyzes the logs to guess why the agent failed.
        """
        if float(entry.get("success_score", 0)) == 1.0:
            return "None (Success)"
            
        steps = entry.get("steps", [])
        if not steps:
            return "Startup Error"
            
        last_step = steps[-1]
        last_action = last_step.get("action", "")
        
        # Failure Heuristics
        if "FAIL" in last_action:
            return "Agent Gave Up (FAIL action)"
        if len(steps) >= 15: # Assuming 15 is max_steps
            return "Timeout (Max Steps Reached)"
        if "WAIT" in last_action and len(steps) > 1:
            # Check if it was waiting repeatedly
            prev_action = steps[-2].get("action", "")
            if "WAIT" in prev_action:
                return "Stuck in Wait Loop"
        
        return "Execution Error / Wrong Action"

    def generate_visuals(self):
        """Generates charts for the report."""
        if self.df.empty: return

        # 1. Success Rate by Domain
        plt.figure(figsize=(10, 6))
        domain_success = self.df.groupby('domain')['success'].mean() * 100
        ax = sns.barplot(x=domain_success.index, y=domain_success.values, palette="viridis")
        ax.set_title("Success Rate by Domain (%)")
        ax.set_ylabel("Success Rate")
        plt.savefig(os.path.join(self.run_dir, "success_by_domain.png"))
        plt.close()

        # 2. Step Distribution (Success vs Fail)
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=self.df, x='success', y='steps', palette="Set2")
        plt.title("Step Count Distribution: Success vs Failure")
        plt.xticks([0, 1], ['Failure', 'Success'])
        plt.savefig(os.path.join(self.run_dir, "steps_distribution.png"))
        plt.close()

    def write_report(self):
        if self.df.empty: return

        success_rate = self.df['success'].mean() * 100
        total = len(self.df)
        passed = self.df['success'].sum()
        
        with open(self.report_path, "w") as f:
            f.write(f"# OSWorld Evaluation Report\n")
            f.write(f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
            
            f.write("## 1. Executive Summary\n")
            f.write(f"- **Total Tasks:** {total}\n")
            f.write(f"- **Solved:** {passed}\n")
            f.write(f"- **Success Rate:** {success_rate:.2f}%\n")
            f.write(f"- **Avg Steps (Success):** {self.df[self.df['success']]['steps'].mean():.1f}\n\n")
            
            f.write("## 2. Failure Mode Analysis\n")
            f.write("Common reasons for failure in this run:\n\n")
            failures = self.df[~self.df['success']]['failure_reason'].value_counts()
            f.write("| Failure Reason | Count |\n|---|---|\n")
            for reason, count in failures.items():
                f.write(f"| {reason} | {count} |\n")
            
            f.write("\n## 3. Domain Breakdown\n")
            domain_stats = self.df.groupby('domain').agg(
                Total=('task_id', 'count'),
                Solved=('success', 'sum')
            )
            domain_stats['Rate'] = (domain_stats['Solved'] / domain_stats['Total'] * 100).round(1)
            f.write(domain_stats.to_markdown())
            
            f.write("\n\n## 4. Recommendations\n")
            f.write("- **Grounding:** If 'Execution Error' is high, consider checking A11y tree truncation logic.\n")
            f.write("- **Planning:** If 'Timeout' is high, the agent may be getting stuck in loops. Consider adding a 'memory' of past actions to the prompt.\n")
            
        print(f"📄 Report generated at: {self.report_path}")
        print(f"📊 Charts saved in: {self.run_dir}")

if __name__ == "__main__":
    # parser = argparse.ArgumentParser()
    # parser.add_argument("--run_dir", required=True, help="Path to the 'run_YYYYMMDD...' directory")
    # args = parser.parse_args()
    
    analyzer = OSWorldAnalyzer('/Users/mvaishak/Developer/AIAgents/myTestAgentV4/results/run_20251121_154544')
    analyzer.load_data()
    analyzer.generate_visuals()
    analyzer.write_report()

🔍 Scanning /Users/mvaishak/Developer/AIAgents/myTestAgentV4/results/run_20251121_130410 for trajectories...
✅ Loaded 1 tasks.
📄 Report generated at: /Users/mvaishak/Developer/AIAgents/myTestAgentV4/results/run_20251121_130410/analysis_report.md
📊 Charts saved in: /Users/mvaishak/Developer/AIAgents/myTestAgentV4/results/run_20251121_130410


/var/folders/rk/f_4tfjj94475mjnm5k79dd1c0000gn/T/ipykernel_58432/1905825321.py:88: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=domain_success.index, y=domain_success.values, palette="viridis")
/var/folders/rk/f_4tfjj94475mjnm5k79dd1c0000gn/T/ipykernel_58432/1905825321.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=self.df, x='success', y='steps', palette="Set2")
